## 0. Imports & Configuration

In [1]:
import json
import os
import re
import math
import random
import pickle
import warnings
from collections import Counter
from typing import List, Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import classification_report, accuracy_score, f1_score

from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)

CFG = {
    # Dataset
    'reviews_per_category': 12000,   # ~12k per category -> ~36k total
    'train_ratio': 0.70,
    'val_ratio':   0.15,
    'test_ratio':  0.15,

    # Preprocessing
    'max_seq_len': 128,              # fixed max sequence length
    'vocab_size': 15000,             # top-N tokens
    'min_freq': 2,                   # minimum token frequency

    # Encoder (Part A)
    'embed_dim': 128,
    'num_heads': 4,
    'num_enc_layers': 3,
    'ff_dim': 256,
    'dropout': 0.1,
    'enc_lr': 3e-4,
    'enc_epochs': 8,
    'enc_batch_size': 64,

    # Retrieval (Part B)
    'top_k': 3,                      # number of retrieved examples

    # Decoder (Part C)
    'num_dec_layers': 2,
    'dec_lr': 3e-4,
    'dec_epochs': 6,
    'dec_batch_size': 32,
    'max_gen_len': 60,               # max tokens to generate
}

print('Configuration loaded.')
print(json.dumps(CFG, indent=2))

Device: cuda
Configuration loaded.
{
  "reviews_per_category": 12000,
  "train_ratio": 0.7,
  "val_ratio": 0.15,
  "test_ratio": 0.15,
  "max_seq_len": 128,
  "vocab_size": 15000,
  "min_freq": 2,
  "embed_dim": 128,
  "num_heads": 4,
  "num_enc_layers": 3,
  "ff_dim": 256,
  "dropout": 0.1,
  "enc_lr": 0.0003,
  "enc_epochs": 8,
  "enc_batch_size": 64,
  "top_k": 3,
  "num_dec_layers": 2,
  "dec_lr": 0.0003,
  "dec_epochs": 6,
  "dec_batch_size": 32,
  "max_gen_len": 60
}


## 1. Dataset Loading

We load three product categories from the Amazon Reviews dataset,  **Sports & Outdoors**, **Cell Phones & Accessories**, and **Beauty**, sampling ~12,000 reviews each for a total of ~36,000. Each sample retains its `reviewText` and `overall` star rating.

In [2]:
def load_category(path: str, n: int, category_name: str) -> List[Dict]:
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            text = obj.get('reviewText', '').strip()
            rating = obj.get('overall', None)
            if text and rating is not None:
                records.append({
                    'text': text,
                    'rating': float(rating),
                    'category': category_name
                })
            if len(records) >= n:
                break
    print(f'  Loaded {len(records):,} reviews from {category_name}')
    return records


FILES = {
    'Sports_and_Outdoors':          'Sports_and_Outdoors_5.json',
    'Cell_Phones_and_Accessories':  'Cell_Phones_and_Accessories_5.json',
    'Beauty':                       'Beauty_5.json',
}

print('Loading dataset...')
all_reviews = []
for cat_name, fname in FILES.items():
    all_reviews.extend(load_category(fname, CFG['reviews_per_category'], cat_name))

random.shuffle(all_reviews)
print(f'\nTotal reviews: {len(all_reviews):,}')

# Category distribution
cat_counts = Counter(r['category'] for r in all_reviews)
print('Category distribution:', dict(cat_counts))

# Rating distribution
rating_counts = Counter(int(r['rating']) for r in all_reviews)
print('Rating distribution:', dict(sorted(rating_counts.items())))

Loading dataset...
  Loaded 12,000 reviews from Sports_and_Outdoors
  Loaded 12,000 reviews from Cell_Phones_and_Accessories
  Loaded 12,000 reviews from Beauty

Total reviews: 36,000
Category distribution: {'Sports_and_Outdoors': 12000, 'Cell_Phones_and_Accessories': 12000, 'Beauty': 12000}
Rating distribution: {1: 2250, 2: 1954, 3: 3439, 4: 6867, 5: 21490}


## 2. Preprocessing Pipeline

### Design Decisions
1. **Text Cleaning** - lower-case, remove HTML tags, strip non-ASCII, collapse whitespace.
2. **Tokenization** - simple whitespace + punctuation split (no NLTK/spaCy dependency).
3. **Vocabulary** - built from the training split only, keeping the top `vocab_size` tokens with frequency ≥ `min_freq`. Special tokens: `<PAD>` (0), `<UNK>` (1), `<BOS>` (2), `<EOS>` (3).
4. **Encoding** - each token string mapped to its integer index.
5. **Padding / Truncation** - all sequences padded or truncated to `max_seq_len`.

In [3]:
def rating_to_sentiment(rating: float) -> int:
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

def text_length_class(text: str) -> int:
    n = len(text.split())
    if n < 50:
        return 0
    elif n <= 150:
        return 1
    else:
        return 2


_HTML_TAG = re.compile(r'<[^>]+>')
_PUNCT    = re.compile(r"([!\"#$%&\'()*+,\-./:;<=>?@[\\\]^_`{|}~])")
_MULTI_SP = re.compile(r'\s+')

def clean_text(text: str) -> str:
    text = text.lower()
    text = _HTML_TAG.sub(' ', text)              # remove HTML
    text = text.encode('ascii', 'ignore').decode()  # strip non-ASCII
    text = _PUNCT.sub(r' \1 ', text)            # space around punctuation
    text = _MULTI_SP.sub(' ', text).strip()     # collapse whitespace
    return text


PAD, UNK, BOS, EOS = '<PAD>', '<UNK>', '<BOS>', '<EOS>'

class Vocabulary:
    def __init__(self, max_size: int = 15000, min_freq: int = 2):
        self.max_size = max_size
        self.min_freq = min_freq
        self.token2id: Dict[str, int] = {}
        self.id2token: Dict[int, str] = {}

    def build(self, texts: List[str]):
        counter = Counter()
        for t in texts:
            counter.update(clean_text(t).split())
        specials = [PAD, UNK, BOS, EOS]
        vocab = specials[:]
        for tok, freq in counter.most_common(self.max_size - len(specials)):
            if freq >= self.min_freq:
                vocab.append(tok)
        self.token2id = {t: i for i, t in enumerate(vocab)}
        self.id2token = {i: t for t, i in self.token2id.items()}
        print(f'  Vocabulary size: {len(self.token2id):,}')

    def encode(self, text: str, max_len: int, add_bos: bool = False, add_eos: bool = False) -> List[int]:
        tokens = clean_text(text).split()
        ids = []
        if add_bos:
            ids.append(self.token2id[BOS])
        for tok in tokens:
            ids.append(self.token2id.get(tok, self.token2id[UNK]))
        if add_eos:
            ids.append(self.token2id[EOS])
        # Truncate (preserve BOS/EOS)
        if len(ids) > max_len:
            if add_eos:
                ids = ids[:max_len - 1] + [self.token2id[EOS]]
            else:
                ids = ids[:max_len]
        # Pad
        ids += [self.token2id[PAD]] * (max_len - len(ids))
        return ids

    def decode(self, ids: List[int], skip_special: bool = True) -> str:
        skip = {self.token2id[t] for t in [PAD, UNK, BOS, EOS]} if skip_special else set()
        return ' '.join(self.id2token.get(i, UNK) for i in ids if i not in skip)

    @property
    def pad_id(self): return self.token2id[PAD]
    @property
    def unk_id(self): return self.token2id[UNK]
    @property
    def bos_id(self): return self.token2id[BOS]
    @property
    def eos_id(self): return self.token2id[EOS]
    def __len__(self):  return len(self.token2id)


n_total = len(all_reviews)
n_train = int(n_total * CFG['train_ratio'])
n_val   = int(n_total * CFG['val_ratio'])
n_test  = n_total - n_train - n_val

train_data = all_reviews[:n_train]
val_data   = all_reviews[n_train:n_train + n_val]
test_data  = all_reviews[n_train + n_val:]

print(f'Train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}')

# Build vocabulary on training texts only
print('Building vocabulary from training data...')
vocab = Vocabulary(max_size=CFG['vocab_size'], min_freq=CFG['min_freq'])
vocab.build([r['text'] for r in train_data])

# Save vocab
with open('results/vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print('Vocabulary saved to results/vocab.pkl')

Train: 25,200 | Val: 5,400 | Test: 5,400
Building vocabulary from training data...
  Vocabulary size: 15,000
Vocabulary saved to results/vocab.pkl


## 3. PyTorch Datasets

In [4]:
class ReviewDataset(Dataset):
    def __init__(self, records, vocab: Vocabulary, max_len: int):
        self.samples = []
        for r in records:
            ids = vocab.encode(r['text'], max_len)
            sent = rating_to_sentiment(r['rating'])
            lenc = text_length_class(r['text'])
            self.samples.append((ids, sent, lenc))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        ids, sent, lenc = self.samples[idx]
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(sent, dtype=torch.long),
            torch.tensor(lenc, dtype=torch.long),
        )


print('Creating encoder datasets...')
enc_train_ds = ReviewDataset(train_data, vocab, CFG['max_seq_len'])
enc_val_ds   = ReviewDataset(val_data,   vocab, CFG['max_seq_len'])
enc_test_ds  = ReviewDataset(test_data,  vocab, CFG['max_seq_len'])

enc_train_loader = DataLoader(enc_train_ds, batch_size=CFG['enc_batch_size'], shuffle=True)
enc_val_loader   = DataLoader(enc_val_ds,   batch_size=CFG['enc_batch_size'], shuffle=False)
enc_test_loader  = DataLoader(enc_test_ds,  batch_size=CFG['enc_batch_size'], shuffle=False)

print(f'Train batches: {len(enc_train_loader)} | Val batches: {len(enc_val_loader)} | Test batches: {len(enc_test_loader)}')

Creating encoder datasets...
Train batches: 394 | Val batches: 85 | Test batches: 85
